# UnlimitedOCR C/Metal demo

Run the local C/Metal OCR engine on `docs/test.png`.

Prerequisites:
- a built `libunlimitedocr.dylib`
- `dist/unlimitedocr-fp16.uocr` exists
- a Metal-capable macOS machine

Library/resource resolution:
- if `UOCR_LIBRARY_PATH` is already set, the notebook uses it
- otherwise it prefers non-debug/precompiled build directories and falls back to `build/debug` only if needed
- if a precompiled `unlimitedocr.metallib` is present, the notebook points Metal resources at it

The first run can take several seconds while Metal resources and model views warm up.


In [ ]:
from pathlib import Path
import os
import sys

ROOT = Path.cwd().resolve()
if not (ROOT / 'pyproject.toml').exists() and (ROOT.parent / 'pyproject.toml').exists():
    ROOT = ROOT.parent.resolve()

SRC_PATH = ROOT / 'src'
MODEL_PATH = ROOT / 'dist' / 'unlimitedocr-fp16.uocr'
IMAGE_PATH = ROOT / 'docs' / 'test.png'

def first_existing(paths):
    for path in paths:
        if path is not None and path.exists():
            return path
    return None

library_from_env = os.environ.get('UOCR_LIBRARY_PATH')
if library_from_env:
    LIBRARY_PATH = Path(library_from_env).expanduser().resolve()
else:
    LIBRARY_PATH = first_existing([
        ROOT / 'build' / 'release' / 'libunlimitedocr.dylib',
        ROOT / 'build' / 'release-qa' / 'libunlimitedocr.dylib',
        ROOT / 'build' / 'precompile' / 'libunlimitedocr.dylib',
        ROOT / 'build' / 'qa-precompiled' / 'libunlimitedocr.dylib',
        ROOT / 'build' / 'metal-no-rt' / 'libunlimitedocr.dylib',
        ROOT / 'build' / 'debug' / 'libunlimitedocr.dylib',
    ])
    if LIBRARY_PATH is None:
        raise FileNotFoundError('could not find libunlimitedocr.dylib; build the project or set UOCR_LIBRARY_PATH')

RESOURCE_PATH = first_existing([
    ROOT / 'build' / 'precompile',
    ROOT / 'build' / 'qa-precompiled',
    ROOT / 'src' / 'backend' / 'metal',
])
if RESOURCE_PATH is None:
    raise FileNotFoundError('could not find Metal resources; build a metallib or use src/backend/metal')

if str(SRC_PATH) not in sys.path:
    sys.path.insert(0, str(SRC_PATH))

# Must be set before importing unlimitedocr_c.ffi / unlimitedocr_c.ocr.
os.environ['UOCR_LIBRARY_PATH'] = str(LIBRARY_PATH)

for path in (LIBRARY_PATH, MODEL_PATH, IMAGE_PATH, RESOURCE_PATH):
    if not path.exists():
        raise FileNotFoundError(path)

print('repo:', ROOT)
print('library:', LIBRARY_PATH)
print('model:', MODEL_PATH)
print('image:', IMAGE_PATH)
print('resources:', RESOURCE_PATH)


## Input image


In [ ]:
from IPython.display import display
from PIL import Image

image = Image.open(IMAGE_PATH)
print(image.size, image.mode)
display(image)


## Run OCR

This uses `preset='base'` and a short generation cap so the demo completes quickly. Increase `max_gen_tokens` for longer outputs.


In [ ]:
from time import perf_counter
from unlimitedocr_c.ocr import ocr_image

start = perf_counter()
result = ocr_image(
    str(IMAGE_PATH),
    model_path=str(MODEL_PATH),
    resource_path=str(RESOURCE_PATH),
    preset='base',
    max_length=4096,
    max_gen_tokens=64,
)
elapsed = perf_counter() - start

print(f'elapsed: {elapsed:.3f}s')
print(f'generated tokens: {len(result.token_ids)}')
print('token ids:', result.token_ids.tolist())
print('\ntext:')
print(result.text)


## Optional: quick one-token smoke

Useful when checking that the engine, model, and Metal resources load correctly.


In [ ]:
start = perf_counter()
smoke = ocr_image(
    str(IMAGE_PATH),
    model_path=str(MODEL_PATH),
    resource_path=str(RESOURCE_PATH),
    preset='base',
    max_length=4096,
    max_gen_tokens=1,
)
print(f'elapsed: {perf_counter() - start:.3f}s')
print(smoke.token_ids.tolist())
print(repr(smoke.text))
